# 11 · FP8 (E4M3 / E5M2) emulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/11_fp8_emulation.ipynb) &nbsp; 📖 Concept deck: [Quantization](https://nilesh-salpe.github.io/visual-ai/inference/07-quantization/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** the two FP8 formats from Micikevicius et al. (2022): enumerate every representable value, round with saturation, and see why per-tensor scaling is needed.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")

print('formats: e4m3 (bias 7, 3 mantissa bits, no infinities) and e5m2 (bias 15, 2 mantissa bits, IEEE-style specials)')

## Task 1: Every representable value

Return a sorted array of all **finite, non-negative** values (including 0):
- Exponent field `e`, mantissa field `m`. Subnormals: `e == 0` → $m/2^{M} \cdot 2^{1-bias}$. Normals: $(1 + m/2^{M}) \cdot 2^{e-bias}$.
- **e4m3**: bias 7, M = 3, e ∈ 0..15, but `e=15, m=7` is NaN (no infinities) → max **448**.
- **e5m2**: bias 15, M = 2, e ∈ 0..30 (e = 31 is reserved for inf/NaN) → max **57,344**.

In [ ]:
def fp8_grid(fmt):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_fp8_grid(f):
    a, b = f('e4m3'), f('e5m2')
    assert len(a) == 127 and len(b) == 124, f"expected 127 (e4m3) and 124 (e5m2) finite non-negative values, got {len(a)}, {len(b)}"
    assert a.max() == 448 and b.max() == 57344, "max normals are 448 (e4m3) and 57,344 (e5m2)"
    assert a[1] == 2.0 ** -9 and b[1] == 2.0 ** -16, "smallest subnormals are 2^-9 and 2^-16"
    assert np.all(np.diff(a) > 0), "grid must be sorted and unique"

check("fp8_grid", _test_fp8_grid, fp8_grid)

## Task 2: Round to FP8 (saturating)

Round every element to the **nearest** representable value (keep the sign) and **saturate** magnitudes beyond the max to ±max.

In [ ]:
def to_fp8(x, fmt, grid=None):
    grid = fp8_grid(fmt) if grid is None else grid
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _grid_ref(fmt):
    bias, M, emax = (7, 3, 15) if fmt == 'e4m3' else (15, 2, 30)
    v = [(m / 2 ** M) * 2.0 ** (1 - bias) if e == 0 else (1 + m / 2 ** M) * 2.0 ** (e - bias)
         for e in range(emax + 1) for m in range(2 ** M) if not (fmt == 'e4m3' and e == 15 and m == 7)]
    return np.array(sorted(v))

def _test_to_fp8(f):
    g = _grid_ref('e4m3')
    assert f(np.array([1000.0, -1000.0]), 'e4m3', g).tolist() == [448.0, -448.0], "saturate to ±448"
    x = np.random.default_rng(0).uniform(0.02, 400, 5000) * np.random.default_rng(1).choice([-1, 1], 5000)
    y = f(x, 'e4m3', g)
    assert np.isin(np.abs(y), g).all(), "outputs must be representable e4m3 values"
    assert (np.abs(y - x) / np.abs(x)).max() <= 2 ** -4 + 1e-12, "relative rounding error in the normal range must be <= 2^-4 (half a step with 3 mantissa bits)"

check("to_fp8", _test_to_fp8, to_fp8)

## Task 3: Per-tensor scaling

Small values underflow without scaling. Implement `scale = amax / fmt_max`, cast `x / scale` to FP8, and return `(dequantized, scale)`.

In [ ]:
def scaled_fp8(x, fmt):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _cast_ref(x, g):
    a = np.minimum(np.abs(x), g[-1]); i = np.clip(np.searchsorted(g, a), 1, len(g) - 1)
    return np.sign(x) * np.where(a - g[i - 1] <= g[i] - a, g[i - 1], g[i])

def _test_scaled_fp8(f):
    x = np.random.default_rng(2).normal(0, 1e-4, 4096)
    gg = np.array(sorted([(m / 8) * 2.0 ** -6 if e == 0 else (1 + m / 8) * 2.0 ** (e - 7) for e in range(16) for m in range(8) if not (e == 15 and m == 7)]))
    naive = _cast_ref(x, gg); y, s = f(x, 'e4m3')
    e = lambda a: np.linalg.norm(a - x) / np.linalg.norm(x)
    assert e(naive) > 0.5, "(sanity) unscaled tiny values mostly underflow"
    assert e(y) < 0.05, f"scaled FP8 error should be < 5%, got {e(y):.3f}"

check("scaled_fp8", _test_scaled_fp8, scaled_fp8)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Every representable value
def fp8_grid_solution(fmt):
    bias, M, emax = (7, 3, 15) if fmt == 'e4m3' else (15, 2, 30)
    vals = []
    for e in range(emax + 1):
        for m in range(2 ** M):
            if fmt == 'e4m3' and e == 15 and m == 7: continue      # NaN
            vals.append((m / 2 ** M) * 2.0 ** (1 - bias) if e == 0 else (1 + m / 2 ** M) * 2.0 ** (e - bias))
    return np.array(sorted(vals))

check("fp8_grid (reference)", _test_fp8_grid, fp8_grid_solution)

In [ ]:
# @title Solution for Task 2: Round to FP8 (saturating)
def to_fp8_solution(x, fmt, grid=None):
    g = fp8_grid_solution(fmt) if grid is None else grid
    a = np.minimum(np.abs(x), g[-1])
    i = np.clip(np.searchsorted(g, a), 1, len(g) - 1)
    lo, hi = g[i - 1], g[i]
    return np.sign(x) * np.where(a - lo <= hi - a, lo, hi)

check("to_fp8 (reference)", _test_to_fp8, to_fp8_solution)

In [ ]:
# @title Solution for Task 3: Per-tensor scaling
def scaled_fp8_solution(x, fmt):
    fmax = 448.0 if fmt == 'e4m3' else 57344.0
    scale = max(np.abs(x).max(), 1e-30) / fmax
    return to_fp8_solution(x / scale, fmt) * scale, scale

check("scaled_fp8 (reference)", _test_scaled_fp8, scaled_fp8_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
for fmt in ('e4m3', 'e5m2'):
    g = fp8_grid_solution(fmt)
    print(f"{fmt}: {len(g)} finite non-negative values, max {g[-1]:g}, smallest subnormal {g[1]:.3g}, relative step ~2^-{3 if fmt=='e4m3' else 2}")
print("e4m3 trades range for precision (often used for weights/activations); e5m2 trades precision for range (often used for gradients).")

---
## Takeaways
- E4M3 has max 448 and more precision; E5M2 has max 57,344 and more range. Micikevicius et al. (2022) reported matching 16-bit training quality on models up to 175B.
- FP8's range is small, so real systems keep **scaling factors** (per tensor or per block) alongside the FP8 data.
- The same idea, small formats plus scales, carries on to FP4 microscaling formats.